# moe-sae discovery **capture** (spec §20 step 4 data) — Colab A100 → GCS

**Capture-only lane** (2026-07-13): the A12 backend map freezes ALL 14 training runs on the TPU/JAX lane — training no longer runs here. This notebook: A7 gate on this runtime → sliced capture → bucket. Then hand off to `merge_in_bucket.py` + the TPU side (`docs/PROCEDURE_DISCOVERY.md`, Phase 3 "Colab variant").

**Disk-safe:** the ~770 GB capture is cut into 12 sequential ~64 GB slices; each is pushed, verified, and wiped before the next (peak disk ≈ 2 slices with the default `CONCURRENCY=2`).

**Rerunnable:** all state lives in GCS — if the session dies, run all cells again and it resumes (≤10 min redone). Works from browser Colab **or the Google Colab VS Code extension** (Select Kernel → Colab → A100).

Setup once: Colab secrets `GITHUB_TOKEN` (repo-read PAT) **and `HF_TOKEN`** (Hugging Face read token — anonymous access gets 403-throttled under sustained load) via the 🔑 panel; set `BUCKET` below to the **us-central2 bucket colocated with the TPU** (storage ~$18–25/mo until Phase 9 teardown; uploads and in-region TPU reads are $0).

⚠️ **Order discipline:** nothing here may run before the OSF registration is posted (Phase 0) — no discovery shard may predate that timestamp.

In [1]:
gpu = !nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
print(gpu)
assert any("A100" in g for g in gpu), "Runtime > Change runtime type > A100 GPU"

['NVIDIA A100-SXM4-80GB, 81920 MiB']


In [ ]:
import os, subprocess

# the us-central2 bucket from scripts/tpu/00_provision.sh (no trailing slash)
os.environ["BUCKET"] = "gs://neural-dragnet-moe-sae"  # e.g. gs://<gcp-project>-moe-sae

# --- GCS auth: browser-Colab popup if available, else gcloud device flow ---
try:
    from google.colab import auth
    auth.authenticate_user()
except Exception:
    if subprocess.run(["gcloud", "auth", "list", "--format=value(account)"],
                      capture_output=True, text=True).stdout.strip() == "":
        # VS Code extension path: paste the code from the printed URL
        !gcloud auth login --no-launch-browser --update-adc

# --- secrets: Colab panel if available, else prompt ------------------------
def secret(name, prompt):
    v = os.environ.get(name)
    if not v:
        try:
            from google.colab import userdata
            v = userdata.get(name)
        except Exception:
            from getpass import getpass
            v = getpass(prompt)
    return v

token = secret("GITHUB_TOKEN", "GitHub PAT (repo read): ")
# HF read token: anonymous Xet/CDN access gets throttled to 403 AccessDenied
# under sustained load (observed 2026-07-13); authenticated requests do not.
os.environ["HF_TOKEN"] = secret("HF_TOKEN", "HF read token: ")

# absolute path: rerunning this cell from inside the repo must NOT re-clone
# (a relative check once nested a second clone at /content/moe-sae/moe-sae)
REPO = "/content/moe-sae"
if not os.path.exists(REPO):
    !git clone -q https://{token}@github.com/andrewbistras/moe-sae.git {REPO}
%cd {REPO}
!git pull -q
!git log --oneline -1

In [ ]:
# Env: keep Colab's CUDA torch; pin transformers for the vendored model code.
!pip -q install -e . --no-deps
!pip -q install "transformers==4.38.2" timm einops sentencepiece safetensors datasets accelerate pyyaml pytest
!python -c "import torch, moe_sae; print('torch', torch.__version__, 'cuda:', torch.cuda.is_available())"

In [ ]:
# Phase 0 (A7 gate + A3, cached in bucket) then sliced capture (12 × 8000
# examples, 2 concurrent). Rerunnable; watch logs/capture_w*.log for progress.
# Training does NOT run here (A12) — next steps are printed at the end.
!bash scripts/cloud/colab_capture.sh

In [ ]:
# Slice progress, straight from the bucket (safe to run from any session).
import json, os, subprocess

B, SLICES = os.environ["BUCKET"], int(os.environ.get("SLICES", 12))
done = 0
for k in range(SLICES):
    r = subprocess.run(
        ["gcloud", "storage", "cat", f"{B}/captures/discovery_50m_w{k}/progress.json"],
        capture_output=True, text=True)
    if r.returncode:
        print(f"w{k:02d}: no checkpoint yet")
        continue
    p = json.loads(r.stdout)
    done += bool(p["complete"])
    print(f"w{k:02d}: complete={p['complete']}  examples={p['captured']}"
          f"  tokens={sum(p.get('tokens', {}).values()):,}")
print(f"\n{done}/{SLICES} slices complete"
      + ("" if done < SLICES else " — run merge_in_bucket.py, then the TPU side"))